# EDA -- music context graphs, tags and text

What this notebook covers:

1. corpus composition (genres, moods, tag frequencies)
2. what a music structure graph actually looks like
3. chord recognition from chroma
4. label sparsity -- the reason the trainers use `pos_weight` and a tuned threshold
5. confirmation that the artist-grouped split really is leak-free

Run `python src/preprocess.py --config configs/mtat.yaml` first (or whichever preset you are using -- set `CONFIG` below to match).

In [ ]:
import sys, json
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import torch
import matplotlib.pyplot as plt

from utils import load_config, load_json, get_device, resolve

CONFIG = "configs/mtat.yaml"    # <-- the preset you preprocessed with, or None for config.yaml

cfg = load_config(REPO / CONFIG if CONFIG else REPO / "config.yaml")
device = get_device(cfg.get("device", "auto"))
processed = resolve(cfg, "processed")
splits = resolve(cfg, "splits")

print("repo     :", REPO)
print("device   :", device, "| source:", cfg["dataset"]["source"],
      "| graph:", cfg["graph"]["kind"], "| segment:", cfg["audio"]["segment_seconds"], "s")
print("processed:", processed)

In [ ]:
index = load_json(processed / "index.json")
meta = load_json(processed / "meta.json")
label_space = load_json(processed / "label_space.json")

print(f"{meta['n_tracks']} tracks | {meta['num_tags']} tags | "
      f"avg {meta['avg_nodes']} nodes, {meta['avg_edges']} edges per graph")
print("splits:", meta["split_sizes"])
index[0]

## 1. Corpus composition

In [ ]:
from collections import Counter

genres = Counter(r["genre"] for r in index if r["genre"])
moods = Counter(r["mood"] for r in index if r.get("mood"))
tag_counts = Counter(meta["tag_counts"])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (title, counter) in zip(axes, [("genre", genres), ("mood", moods),
                                       ("top-15 tags", dict(tag_counts.most_common(15)))]):
    keys = list(counter)
    ax.barh(keys, [counter[k] for k in keys])
    ax.set_title(title)
    ax.invert_yaxis()
plt.tight_layout()

## 2. A music structure graph\n\nNodes are time segments, their length set by `audio.segment_seconds`. Edges are temporal adjacency plus chroma/MFCC similarity above tau, with a size-scaled k-NN backstop. **Watch the density line**: above ~0.5 the graph is close to complete and message passing degenerates towards mean pooling.

In [ ]:
from graph_builder import EDGE_TYPE_NAMES, graph_summary

track_id = index[0]["track_id"]
g = torch.load(processed / "graphs" / f"{track_id}.pt", weights_only=False)
print(json.dumps(graph_summary(g), indent=2))
print("\ncaption:", g.text)

In [ ]:
# Draw it: nodes on a circle, edges coloured by type.
import matplotlib.patches as mpatches

n = int(g.num_nodes)
angles = np.linspace(0, 2 * np.pi, n, endpoint=False)
pos = np.stack([np.cos(angles), np.sin(angles)], axis=1)
colors = {0: "#3377cc", 1: "#cc7733", 2: "#33aa66", 3: "#cccccc"}

fig, ax = plt.subplots(figsize=(6, 6))
src, dst = g.edge_index.numpy()
for s, d, t in zip(src, dst, g.edge_type.tolist()):
    if s == d:
        continue
    ax.plot(*zip(pos[s], pos[d]), color=colors.get(t, "#999"), alpha=0.6, lw=1.4)
ax.scatter(pos[:, 0], pos[:, 1], s=420, c="white", edgecolors="black", zorder=3)
for i, (x, y) in enumerate(pos):
    ax.text(x, y, str(i), ha="center", va="center", zorder=4, fontsize=9)
ax.legend(handles=[mpatches.Patch(color=c, label=EDGE_TYPE_NAMES[t])
                   for t, c in colors.items() if t != 3], loc="upper right", fontsize=8)
ax.set_title(f"segment graph -- {track_id}")
ax.set_aspect("equal"); ax.axis("off")

## 3. Chord recognition from chroma\n\nEach segment's 12-bin chroma vector is matched against 24 triad templates.

In [ ]:
from data.synthetic import synthesize_track
from audio_features import extract_track_features, chromagram
from graph_builder import recognise_chords

y, truth = synthesize_track("jazz", "calm", seed=7, duration=20.0)
tf = extract_track_features(y, "demo", cfg)
idx, names = recognise_chords(tf.chroma_segments)

print("ground-truth progression:", truth["chord_sequence"])
print("recognised from chroma :", [names[i] for i in idx])

fig, ax = plt.subplots(figsize=(11, 3))
im = ax.imshow(chromagram(y, cfg["audio"]["sample_rate"]), aspect="auto", origin="lower",
               cmap="magma")
ax.set_yticks(range(12))
ax.set_yticklabels(["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"])
ax.set_xlabel("frame"); ax.set_title("chromagram")
plt.colorbar(im, ax=ax)

## 4. Label sparsity\n\nMulti-label targets are mostly zeros, which is why a flat 0.5 threshold is a bad operating point.

In [ ]:
Y = np.stack([np.isin(np.arange(len(label_space["tags"])),
                      [label_space["tags"].index(t) for t in r["tags"]
                       if t in label_space["tags"]]).astype(float)
              for r in index])

print(f"label density: {Y.mean():.3f}  ({Y.sum(axis=1).mean():.1f} tags per track)")
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(Y.sum(axis=1), bins=range(int(Y.sum(axis=1).max()) + 2), edgecolor="k")
axes[0].set_xlabel("tags per track"); axes[0].set_title("tags per track")
axes[1].bar(range(Y.shape[1]), sorted(Y.sum(axis=0), reverse=True))
axes[1].set_xlabel("tag (sorted)"); axes[1].set_ylabel("support")
axes[1].set_title("tag support -- long tail")
plt.tight_layout()

## 5. Split integrity

In [ ]:
split_ids = {name: set(load_json(splits / f"{name}.json"))
             for name in ("train", "val", "test")}
by_id = {r["track_id"]: r for r in index}
artists = {name: {by_id[t]["artist"] for t in ids if t in by_id}
           for name, ids in split_ids.items()}

print({name: len(ids) for name, ids in split_ids.items()})
print("strategy:", meta.get("split_strategy"))
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    shared = artists[a] & artists[b]
    print(f"{a} & {b}: {len(shared)} shared artists"
          + (f"  <-- LEAKAGE: {sorted(shared)[:5]}" if shared else "  OK"))